# Rékolte — Notebook 2: Feature Engineering, Model Training & Evaluation
**Project:** Predicting Sugarcane Yield in Mauritius Using Satellite Imagery  
**Student:** Yashvin Booputh (M01006629)  
**Module:** CST3990 — Spring 2025/2026

---

## What this notebook does
1. Merges bulletin harvest data with GEE satellite + climate features
2. Engineers the full feature matrix (31 features per row)
3. Trains Random Forest and XGBoost with Leave-One-Season-Out (LOSO) cross-validation
4. Tunes hyperparameters using grid search
5. Evaluates both models on the 2025 holdout season
6. Saves trained models and metadata to Google Drive

### Feature matrix
| Features | Count |
|----------|-------|
| Monthly NDVI (June–December) | 7 |
| Season NDVI aggregates (mean, max, cumulative) | 3 |
| Region one-hot (EST, NORD, OUEST, SUD vs CENTRE baseline) | 4 |
| Satellite sensor one-hot (landsat8, sentinel2 vs landsat7 baseline) | 2 |
| Monthly CHIRPS rainfall (June–December) | 7 |
| Monthly ERA5-Land temperature (June–December) | 7 |
| Previous season harvested area (survivor effect) | 1 |
| **Total** | **31** |

> **Note:** `season` (year) intentionally excluded — it acts as a time-trend shortcut
> that crowds out NDVI/climate signals and inflates LOSO R² artificially.

### Training / evaluation split
- **Training:** 2008–2024 (18 seasons × 5 regions = 90 rows) with LOSO CV
- **Holdout test:** 2025 (5 rows — never seen during training or tuning)


## Cell 1 — Install packages

In [ ]:
!pip install xgboost scikit-learn joblib --quiet
print('Packages ready.')

## Cell 2 — Imports & Drive mount

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE  = '/content/drive/MyDrive/CST3990 - Final Year Project'
DATA_DIR    = f'{DRIVE_BASE}/data'
MODEL_DIR   = f'{DRIVE_BASE}/model'
os.makedirs(MODEL_DIR, exist_ok=True)

print('Imports OK. Drive mounted.')

## Cell 3 — Load Data

In [ ]:
# Harvest bulletin data (90 rows: 18 seasons × 5 regions)
df_harvest = pd.read_csv(f'{DATA_DIR}/season_end_data.csv')
print(f'Harvest data: {df_harvest.shape}')
print(df_harvest[['season','region','tch']].head(10))

print()

# Satellite NDVI features (90 rows: 18 seasons × 5 regions)
df_sat = pd.read_csv(f'{MODEL_DIR}/satellite_features.csv')
print(f'Satellite features: {df_sat.shape}')
print(df_sat[['season','region','satellite','ndvi_mean','observation_count']].head(10))

## Cell 4 — Merge Datasets

In [ ]:
df = pd.merge(
    df_harvest[['season', 'region', 'tch', 'surface_harvested', 'cane_production']],
    df_sat,
    on=['season', 'region'],
    how='inner'
)

print(f'Merged dataset: {df.shape}')
print(f'Seasons: {sorted(df["season"].unique())}')
print(f'Regions: {sorted(df["region"].unique())}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

# Save merged features
df.to_csv(f'{MODEL_DIR}/merged_features.csv', index=False)
print(f'\nSaved merged_features.csv ({len(df)} rows)')

## Cell 5 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. TCH distribution
axes[0, 0].hist(df['tch'], bins=20, color='#2d5016', edgecolor='white')
axes[0, 0].set_title('TCH distribution (all regions/seasons)')
axes[0, 0].set_xlabel('TCH (tonnes cane / ha)')

# 2. TCH by region
regions = ['NORD', 'SUD', 'EST', 'OUEST', 'CENTRE']
tch_by_region = [df[df['region'] == r]['tch'].values for r in regions]
axes[0, 1].boxplot(tch_by_region, labels=regions, patch_artist=True)
axes[0, 1].set_title('TCH by region')
axes[0, 1].set_ylabel('TCH')

# 3. TCH over time
for region in regions:
    sub = df[df['region'] == region].sort_values('season')
    axes[0, 2].plot(sub['season'], sub['tch'], marker='o', label=region, linewidth=1.5)
axes[0, 2].set_title('TCH trends by region (2008–2025)')
axes[0, 2].set_xlabel('Season')
axes[0, 2].legend(fontsize=8)
axes[0, 2].tick_params(axis='x', rotation=45)

# 4. NDVI mean vs TCH
axes[1, 0].scatter(df['ndvi_mean'], df['tch'], c='#C8891A', alpha=0.7)
axes[1, 0].set_xlabel('NDVI mean (season)')
axes[1, 0].set_ylabel('TCH')
axes[1, 0].set_title('NDVI mean vs TCH')

# 5. NDVI cumulative vs TCH
axes[1, 1].scatter(df['ndvi_cumulative'], df['tch'], c='#2d5016', alpha=0.7)
axes[1, 1].set_xlabel('NDVI cumulative')
axes[1, 1].set_ylabel('TCH')
axes[1, 1].set_title('NDVI cumulative vs TCH')

# 6. Correlation heatmap (NDVI features + TCH)
ndvi_cols = [c for c in df.columns if c.startswith('ndvi_')]
corr_cols = ndvi_cols + ['season', 'tch']
corr = df[corr_cols].corr()
sns.heatmap(corr[['tch']].sort_values('tch', ascending=False),
            annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, ax=axes[1, 2],
            cbar_kws={'shrink': 0.8})
axes[1, 2].set_title('Correlation with TCH')

plt.suptitle('Exploratory Data Analysis — Rékolte', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/eda_plots.png', dpi=150)
plt.show()

## Cell 6 — Feature Engineering

1. Impute missing monthly NDVI values with that row’s `ndvi_mean`
   (only affects months with zero cloud-free observations)
2. Impute any missing `rainfall_*` or `temp_*` values with the column mean
   (ERA5/CHIRPS are complete for 2008–2025 — this is a safety net only)
3. One-hot encode `region` (CENTRE = baseline, dropped)
4. One-hot encode `satellite` sensor (Landsat 7 = baseline, dropped)
   Tells the model that NDVI scales differ between Landsat 7, Landsat 8, and Sentinel-2
5. Split into train (2008–2024, 85 rows) and holdout test (2025, 5 rows)

In [ ]:
# ── Monthly column lists ─────────────────────────────────────────────────────
MONTH_SUFFIX = ['june', 'july', 'aug', 'sep', 'oct', 'nov', 'dec']
ndvi_monthly     = [f'ndvi_{m}'     for m in MONTH_SUFFIX]
rainfall_monthly = [f'rainfall_{m}' for m in MONTH_SUFFIX]
temp_monthly     = [f'temp_{m}'     for m in MONTH_SUFFIX]

# ── Impute missing monthly NDVI ───────────────────────────────────────────────
for col in ndvi_monthly:
    missing = df[col].isnull().sum()
    if missing > 0:
        df[col] = df[col].fillna(df['ndvi_mean'])
        print(f'Imputed {missing} NDVI NaN in {col} with row ndvi_mean')

# ── Impute missing rainfall / temperature (safety net) ────────────────────────
for col in rainfall_monthly + temp_monthly:
    missing = df[col].isnull().sum()
    if missing > 0:
        col_mean = df[col].mean()
        df[col] = df[col].fillna(col_mean)
        print(f'Imputed {missing} NaN in {col} with column mean {col_mean:.2f}')


# ── Add surface_harvested_prev_season ────────────────────────────────────────
# Captures the survivor effect: as weaker estates exit production, the
# remaining land is higher quality -> higher TCH. Using PREVIOUS season's
# area avoids any leakage (it's known before the current season ends).
df = df.sort_values(['region', 'season']).reset_index(drop=True)
df['surface_prev'] = df.groupby('region')['surface_harvested'].shift(1)

# Impute 2008 (no prior season) with that region's own 2008 surface value
mask_2008 = df['season'] == 2008
df.loc[mask_2008, 'surface_prev'] = df.loc[mask_2008, 'surface_harvested']
print(f'surface_prev: {df["surface_prev"].isnull().sum()} missing values after imputation')
print(df[['season','region','surface_harvested','surface_prev']].head(10))
# ── One-hot encode region (CENTRE = baseline) ─────────────────────────────────
# Preserve region labels before get_dummies drops the column
df['reg_name'] = df['region']
df_encoded = pd.get_dummies(df, columns=['region'], drop_first=True, dtype=int)
region_dummies = sorted([c for c in df_encoded.columns if c.startswith('region_')])  # define BEFORE adding region_label
df_encoded['reg_name'] = df['reg_name']  # carry through for results display
print(f'Region dummies: {region_dummies}')

# ── One-hot encode satellite sensor (Landsat 7 = baseline) ───────────────────
# Tells the model NDVI magnitude differs between sensors:
#   Landsat 7 ETM+ / Landsat 8 OLI / Sentinel-2 SR have different band widths
df_encoded = pd.get_dummies(df_encoded, columns=['satellite'], drop_first=True, dtype=int)
sat_dummies = sorted([c for c in df_encoded.columns if c.startswith('satellite_')])
print(f'Satellite dummies: {sat_dummies}')

# ── Define feature matrix ─────────────────────────────────────────────────────
NDVI_FEATURES     = ndvi_monthly + ['ndvi_mean', 'ndvi_max', 'ndvi_cumulative']
RAINFALL_FEATURES = rainfall_monthly
TEMP_FEATURES     = temp_monthly

ALL_FEATURES = (
    NDVI_FEATURES        # 10: monthly NDVI + season aggregates
    + region_dummies     #  4: EST, NORD, OUEST, SUD (CENTRE baseline)
    + sat_dummies        #  2: landsat8, sentinel2 (landsat7 baseline)
    + RAINFALL_FEATURES  #  7: CHIRPS monthly rainfall (mm)
    + TEMP_FEATURES      #  7: ERA5 monthly temperature (degC)
    # season removed: forces model to use NDVI/climate signals, not year trend
    + ['surface_prev']  #  1: prev season harvested area (ha) — survivor effect
)
TARGET = 'tch'

print(f'\nTotal features: {len(ALL_FEATURES)}')
print('Feature groups:')
print(f'  NDVI:        {len(NDVI_FEATURES)}')
print(f'  Region:      {len(region_dummies)}')
print(f'  Satellite:   {len(sat_dummies)}')
print(f'  Rainfall:    {len(RAINFALL_FEATURES)}')
print(f'  Temperature: {len(TEMP_FEATURES)}')
print(f'  (season removed)')

# ── Train / holdout split ─────────────────────────────────────────────────────
train_df = df_encoded[df_encoded['season'] <= 2024].copy()
test_df  = df_encoded[df_encoded['season'] == 2025].copy()

X_train       = train_df[ALL_FEATURES].values
y_train       = train_df[TARGET].values
X_test        = test_df[ALL_FEATURES].values
y_test        = test_df[TARGET].values
train_seasons = train_df['season'].values

print(f'\nTraining set: {X_train.shape}  ({len(train_df["season"].unique())} seasons, 2008–2024)')
print(f'Test set:     {X_test.shape}   (2025 holdout)')

## Cell 7 — Leave-One-Season-Out (LOSO) Cross-Validation Setup

Standard k-fold CV would leak future seasons into training folds — wrong for time-structured data.  
LOSO holds out one entire season at a time:
- Train on 16 seasons → predict 1 held-out season → rotate (17 iterations)
- This simulates realistic "predict next season" performance.

In [ ]:
def loso_evaluate(model, X, y, seasons_array):
    """
    Leave-One-Season-Out evaluation.
    Returns dict with per-fold predictions and overall RMSE, MAE, R².
    """
    unique_seasons = sorted(np.unique(seasons_array))
    all_preds  = np.zeros(len(y))
    all_actual = y.copy()

    for held_out in unique_seasons:
        train_idx = seasons_array != held_out
        test_idx  = seasons_array == held_out

        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[test_idx])
        all_preds[test_idx] = preds

    rmse = np.sqrt(mean_squared_error(all_actual, all_preds))
    mae  = mean_absolute_error(all_actual, all_preds)
    r2   = r2_score(all_actual, all_preds)

    return {
        'predictions': all_preds,
        'actual':      all_actual,
        'rmse': rmse,
        'mae':  mae,
        'r2':   r2
    }

# Seasons array for LOSO
train_seasons = train_df['season'].values
print('LOSO setup ready.')
print(f'Seasons in training set: {sorted(np.unique(train_seasons))}')

## Cell 8 — Random Forest: Grid Search + LOSO Evaluation

Grid search finds the best hyperparameters.  
We use LOSO CV as the scoring method so the grid search itself is cross-validated properly.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
logo = LeaveOneGroupOut()

# ── Random Forest: hyperparameter grid ───────────────────────────────────────
rf_param_grid = {
    'n_estimators':      [100, 300, 500],
    'max_depth':         [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.5]
}

rf_base = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)
rf_grid = GridSearchCV(
    rf_base,
    rf_param_grid,
    cv=logo,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
rf_grid.fit(X_train, y_train, groups=train_seasons)

best_rf_params = rf_grid.best_params_
print(f'Best RF params: {best_rf_params}')
print(f'Best CV RMSE:   {-rf_grid.best_score_:.4f}')

# ── LOSO evaluation with best params ─────────────────────────────────────────
best_rf = RandomForestRegressor(**best_rf_params, random_state=RANDOM_SEED, n_jobs=-1)
rf_loso = loso_evaluate(best_rf, X_train, y_train, train_seasons)

print(f'\nRandom Forest — LOSO CV Results:')
print(f'  RMSE: {rf_loso["rmse"]:.4f}')
print(f'  MAE:  {rf_loso["mae"]:.4f}')
print(f'  R\u00b2:   {rf_loso["r2"]:.4f}')


## Cell 9 — XGBoost: Grid Search + LOSO Evaluation

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
logo = LeaveOneGroupOut()

xgb_param_grid = {
    'n_estimators':  [100, 300, 500],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample':     [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5]
}

xgb_base = XGBRegressor(
    random_state=RANDOM_SEED,
    tree_method='hist',
    verbosity=0
)
xgb_grid = GridSearchCV(
    xgb_base,
    xgb_param_grid,
    cv=logo,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
xgb_grid.fit(X_train, y_train, groups=train_seasons)

best_xgb_params = xgb_grid.best_params_
print(f'\nBest XGBoost params: {best_xgb_params}')
print(f'Best CV RMSE:        {-xgb_grid.best_score_:.4f}')

best_xgb = XGBRegressor(**best_xgb_params, random_state=RANDOM_SEED, tree_method='hist', verbosity=0)
xgb_loso = loso_evaluate(best_xgb, X_train, y_train, train_seasons)

print(f'\nXGBoost — LOSO CV Results:')
print(f'  RMSE: {xgb_loso["rmse"]:.4f}')
print(f'  MAE:  {xgb_loso["mae"]:.4f}')
print(f'  R²:   {xgb_loso["r2"]:.4f}')

## Cell 10 — Compare LOSO CV Results

In [ ]:
print('=' * 50)
print(f'{"Metric":<12} {"Random Forest":>18} {"XGBoost":>18}')
print('=' * 50)
for metric in ['rmse', 'mae', 'r2']:
    print(f'{metric.upper():<12} {rf_loso[metric]:>18.4f} {xgb_loso[metric]:>18.4f}')
print('=' * 50)

# Scatter: predicted vs actual (LOSO) — both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, loso, label, color in zip(
    axes,
    [rf_loso, xgb_loso],
    ['Random Forest', 'XGBoost'],
    ['#2d5016', '#C8891A']
):
    lo = min(loso['actual'].min(), loso['predictions'].min()) - 2
    hi = max(loso['actual'].max(), loso['predictions'].max()) + 2
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, alpha=0.5, label='Perfect')
    ax.scatter(loso['actual'], loso['predictions'], color=color, alpha=0.7, s=50)
    ax.set_xlabel('Actual TCH')
    ax.set_ylabel('Predicted TCH')
    ax.set_title(f'{label} LOSO  R\u00b2={loso["r2"]:.4f}')
    ax.legend()
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/loso_predicted_vs_actual.png', dpi=150)
plt.show()


## Cell 11 — Train Final Models on Full 2008–2024 Training Set

In [ ]:
# ── Train final models on full 2008–2024 training set ───────────────────────
final_xgb = XGBRegressor(**best_xgb_params, random_state=RANDOM_SEED,
                          tree_method='hist', verbosity=0)
final_xgb.fit(X_train, y_train)
print('Final XGBoost trained on 2008–2024.')

final_rf = RandomForestRegressor(**best_rf_params, random_state=RANDOM_SEED, n_jobs=-1)
final_rf.fit(X_train, y_train)
print('Final Random Forest trained on 2008–2024.')


## Cell 12 — Evaluate on 2025 Holdout
These 5 rows (one per region) were **never seen** during training or hyperparameter tuning.

In [ ]:
xgb_pred_2025 = final_xgb.predict(X_test)
rf_pred_2025  = final_rf.predict(X_test)

results_2025 = pd.DataFrame({
    'region':        test_df['reg_name'].values,
    'actual_tch':    y_test,
    'xgb_predicted': xgb_pred_2025,
    'xgb_error':     xgb_pred_2025 - y_test,
    'rf_predicted':  rf_pred_2025,
    'rf_error':      rf_pred_2025  - y_test,
})

print('2025 Holdout Results:')
print(results_2025.to_string(index=False, float_format='%.2f'))

print('\n2025 Holdout Metrics:')
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred_2025))
xgb_mae  = mean_absolute_error(y_test, xgb_pred_2025)
xgb_r2   = r2_score(y_test, xgb_pred_2025)
rf_rmse  = np.sqrt(mean_squared_error(y_test, rf_pred_2025))
rf_mae   = mean_absolute_error(y_test, rf_pred_2025)
rf_r2    = r2_score(y_test, rf_pred_2025)

print(f'  XGBoost   RMSE={xgb_rmse:.4f}  MAE={xgb_mae:.4f}  R\u00b2={xgb_r2:.4f}')
print(f'  RF        RMSE={rf_rmse:.4f}  MAE={rf_mae:.4f}  R\u00b2={rf_r2:.4f}')

xgb_2025_r2 = xgb_r2
rf_2025_r2  = rf_r2

results_2025.to_csv(f'{MODEL_DIR}/predictions_2025.csv', index=False)
print('\nSaved predictions_2025.csv')


## Cell 13 — 2025 Holdout Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(len(results_2025))
width = 0.25

for ax, pred_col, r2_val, label, color in zip(
    axes,
    ['xgb_predicted', 'rf_predicted'],
    [xgb_2025_r2, rf_2025_r2],
    ['XGBoost', 'Random Forest'],
    ['#C8891A', '#2d5016']
):
    ax.bar(x - width/2, results_2025['actual_tch'], width, label='Actual',    color='#888', alpha=0.7)
    ax.bar(x + width/2, results_2025[pred_col],     width, label='Predicted', color=color,  alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(results_2025['region'])
    ax.set_ylabel('TCH (tonnes cane / ha)')
    ax.set_title(f'{label} — 2025 Holdout  R\u00b2={r2_val:.4f}')
    ax.legend()
    ax.set_ylim(0, max(results_2025['actual_tch'].max(), results_2025[pred_col].max()) * 1.15)
    for i, (actual, pred) in enumerate(zip(results_2025['actual_tch'], results_2025[pred_col])):
        err = pred - actual
        ax.text(i + width/2, pred + 0.5, f'{err:+.1f}', ha='center', fontsize=8, color=color)

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/predictions_2025_bar.png', dpi=150)
plt.show()


## Cell 14 — Feature Importance
Which NDVI months and features matter most for predicting TCH?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

for ax, model, label, color in zip(
    axes,
    [final_xgb, final_rf],
    ['XGBoost', 'Random Forest'],
    ['#C8891A', '#2d5016']
):
    imp = pd.Series(model.feature_importances_, index=ALL_FEATURES).sort_values(ascending=True)
    bars = ax.barh(imp.index, imp.values, color=color, alpha=0.85)
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'{label} Feature Importance')
    for bar, val in zip(bars, imp.values):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=7)

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/feature_importance.png', dpi=150)
plt.show()

print('Top 5 features (XGBoost):')
xgb_imp = pd.Series(final_xgb.feature_importances_, index=ALL_FEATURES).sort_values(ascending=False)
for feat, imp in xgb_imp.head(5).items():
    print(f'  {feat:<30} {imp:.6f}')

print('\nTop 5 features (Random Forest):')
rf_imp = pd.Series(final_rf.feature_importances_, index=ALL_FEATURES).sort_values(ascending=False)
for feat, imp in rf_imp.head(5).items():
    print(f'  {feat:<30} {imp:.6f}')


## Cell 15 — Save Models and Summary

- **Random Forest:** saved with `joblib` (sklearn serialization is version-stable)
- **XGBoost:** saved with `get_booster().save_model()` in XGBoost native `.ubj` format.
  This is version-stable — unlike joblib-wrapped XGBRegressor which breaks when the
  XGBoost version changes between Colab and Render.

In [ ]:
import json as json_lib

RF_PATH  = f'{MODEL_DIR}/rf_model.joblib'
XGB_PATH = f'{MODEL_DIR}/xgb_model.ubj'

joblib.dump(final_rf, RF_PATH)
print(f'Saved RF:  {RF_PATH}')

final_xgb.get_booster().save_model(XGB_PATH)
print(f'Saved XGB: {XGB_PATH}')

xgb_2025_rmse = float(np.sqrt(mean_squared_error(y_test, xgb_pred_2025)))
rf_2025_rmse  = float(np.sqrt(mean_squared_error(y_test, rf_pred_2025)))

metadata = {
    'feature_columns': ALL_FEATURES,
    'target': TARGET,
    'training_seasons': list(range(2008, 2025)),
    'test_season': 2025,
    'feature_groups': {
        'ndvi':         NDVI_FEATURES,
        'region':       region_dummies,
        'satellite':    sat_dummies,
        'rainfall':     RAINFALL_FEATURES,
        'temperature':  TEMP_FEATURES,
        'surface_prev': ['surface_prev'],
        'season':       '(removed — time-trend proxy)',
    },
    'climate_sources': {
        'rainfall':    'CHIRPS Daily (UCSB-CHG/CHIRPS/DAILY), ~5km',
        'temperature': 'ERA5-Land Monthly (ECMWF/ERA5_LAND/MONTHLY_AGGR), ~11km',
    },
    'random_seed': RANDOM_SEED,
    'models': {
        'xgboost': {
            'path':           XGB_PATH,
            'params':         best_xgb_params,
            'loso_rmse':      float(xgb_loso['rmse']),
            'loso_mae':       float(xgb_loso['mae']),
            'loso_r2':        float(xgb_loso['r2']),
            'test_2025_rmse': xgb_2025_rmse,
            'test_2025_r2':   float(xgb_2025_r2),
        },
        'random_forest': {
            'path':           RF_PATH,
            'params':         best_rf_params,
            'loso_rmse':      float(rf_loso['rmse']),
            'loso_mae':       float(rf_loso['mae']),
            'loso_r2':        float(rf_loso['r2']),
            'test_2025_rmse': rf_2025_rmse,
            'test_2025_r2':   float(rf_2025_r2),
        },
    },
}

with open(f'{MODEL_DIR}/model_metadata.json', 'w') as fh:
    json_lib.dump(metadata, fh, indent=2)
print(f'Saved metadata: {MODEL_DIR}/model_metadata.json')

print('\n' + '=' * 55)
print('FINAL SUMMARY — RF + XGBoost + surface_prev')
print('=' * 55)
print(f'{"Metric":<25} {"Random Forest":>14} {"XGBoost":>14}')
print('-' * 55)
print(f'{"LOSO RMSE":<25} {rf_loso["rmse"]:>14.4f} {xgb_loso["rmse"]:>14.4f}')
print(f'{"LOSO MAE":<25} {rf_loso["mae"]:>14.4f} {xgb_loso["mae"]:>14.4f}')
print(f'{"LOSO R\u00b2":<25} {rf_loso["r2"]:>14.4f} {xgb_loso["r2"]:>14.4f}')
print(f'{"2025 Holdout RMSE":<25} {rf_2025_rmse:>14.4f} {xgb_2025_rmse:>14.4f}')
print(f'{"2025 Holdout R\u00b2":<25} {rf_2025_r2:>14.4f} {xgb_2025_r2:>14.4f}')
print('=' * 55)
print(f'Features used: {len(ALL_FEATURES)} (including surface_prev)')
print(f'Training rows: {X_train.shape[0]}')


## Cell 16 — Files saved to Google Drive
Verify everything is in place before proceeding to the Flask API.

In [ ]:
print(f'Files in {MODEL_DIR}:')
for f in sorted(os.listdir(MODEL_DIR)):
    size_kb = os.path.getsize(f'{MODEL_DIR}/{f}') / 1024
    print(f'  {f:<40}  {size_kb:>8.1f} KB')

print('\nNext step: Flask API — load rf_model.joblib + xgb_model.joblib, serve /predict endpoint.')